# 02 - Credit Lifecycle

Credits flow through a lifecycle: they are added, charged, and sometimes refunded. Understanding this lifecycle is critical to building reliable credit systems.

ducto's core operations are simple:
- `add_credits()` deposits credits into a user's account.
- `deduct()` charges atomically — calculates the cost and debits in one transaction.
- `refund_credits()` reverses a completed deduction.

For concurrent or multi-step operations, ducto provides the **lease lifecycle**: `reserve()` places an atomic hold against `available = balance − Σ(active holds)`, preventing overdraft under concurrency. When the work finishes, `settle()` charges the actual cost (freeing any unused portion of the hold), or `release()` cancels the hold without charging. This is the admission-control pattern for concurrent ops — see notebook **11 - Financial Safety** for the full walkthrough.

In [ ]:
from datetime import datetime, timedelta
from ducto.interface.postgres import PostgresStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PlanDefinition,
    CreditMetadata,
)
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
import uuid
print("✔ PostgresStore ready.")

### Add credits

The first operation in any credit lifecycle is adding credits to a user's account. `PostgresStore.add_credits()` creates a new credit entry with a unique transaction ID and returns the user's updated balance.

Each deposit has a `type` label - such as `"signup_bonus"`, `"purchase"`, or `"adjustment"` - which serves as an audit category. This makes it possible to later query how many credits came from signups versus purchases versus manual adjustments.

In [ ]:
# Generate a unique user ID for this demonstration.
# In production, this would be the user's UUID from your auth system.
user = str(uuid.uuid4())

# Deposit 10,000 credits as a signup bonus into the user's account.
# The add_credits() method returns an AddCreditsResult object containing:
# - transaction_id: a unique identifier for this deposit, used for audit trails and future refunds
# - new_balance: the user's total credit balance after the deposit completes
r = store.add_credits(user, 10_000, type="signup_bonus")
print(f"  Tx:        {r.transaction_id}")  # Unique reference for this deposit
print(f"  Balance:   {r.new_balance}")  # User now has 10,000 credits available to spend

### Refund a deduction

Sometimes a completed deduction needs to be reversed. For example, if a credit purchase fails after the initial authorization, or if a customer requests a refund for a faulty model response.

`refund_credits()` restores the deducted amount to the user's balance. Critically, it requires the original deduction's `transaction_id` as a reference. This ensures a proper audit trail: the refund is linked to the original spend, and the same transaction cannot be refunded twice. The original deduction's transaction record remains in the database unchanged - it is not deleted or modified - preserving a complete history of the spend-and-refund cycle for auditing purposes.

In [ ]:
# Refund the deduction we just completed, referencing its original transaction_id.
# Passing the deduction's transaction_id allows the system to:
# 1. Validate that the referenced transaction exists and has not already been refunded
# 2. Create an audit trail linking the refund to the original spend
# 3. Prevent double-reversal (deduplication) — the same transaction cannot be refunded twice
ref = store.refund_credits(ded.transaction_id, amount=2_000, reason="test")
print(f"  Refund tx:   {ref.refund_transaction_id}")  # New unique ID for this refund operation
print(f"  New balance: {ref.new_balance}")  # Balance restored to 10,000 — the original deposit amount
assert ref.new_balance == 10_000  # Balance fully restored after refunding the full 2,000 credits

### The lease lifecycle (`reserve` → `settle` → `release`)

For concurrent or multi-step operations, use the **lease lifecycle** on `CreditManager`:

- `reserve` places an atomic hold against `available = balance − Σ(active holds)`, so concurrent operations actually see each other. It is the *only* admission gate (supporting `max_concurrent`, feature gating, and `strict_prepaid` / `overdraft` presets).
- `settle` charges the **actual** cost when the work finishes — so it bills the real amount even if it differs from the hold, and frees any unused portion.
- `release` cancels the hold without charging.

In `strict_prepaid` mode (the default) this gives a structural zero-debt guarantee: size the hold at the worst case and the balance can never go negative. See notebook **11 - Financial Safety** for the full walkthrough (overdraft, double-submit protection, reload hooks, advisory reads). A minimal version:

In [ ]:
from decimal import Decimal

# A manager over the same store; strict_prepaid is the safe default.
lease_mgr = CreditManager(store=store, policy="strict_prepaid")
lease_mgr.publish_pricing_from_dict({"models": {"_default": "input_tokens * 1"}, "min_balance": 0})

lease_user = str(uuid.uuid4())
lease_mgr.add_credits(lease_user, Decimal("100"), "signup_bonus")

# Reserve the WORST-CASE hold, do the work, then settle the ACTUAL cost.
lease = lease_mgr.reserve(lease_user, Decimal("40"))                 # hold 40 against available
print(f"  Held {lease.amount}, available now {lease.available}")     # 100 - 40 = 60
ded2 = lease_mgr.settle(lease_user, lease.lease_id, Decimal("11"))   # actual cost only 11
print(f"  Charged {ded2.amount}, balance {ded2.balance_after}")      # 100 - 11 = 89
assert ded2.balance_after == Decimal("89")  # the unused 29 hold was freed automatically

In [ ]:
cleanup(pgdata)